# 01 - Data cleaning

**Goal:** understand the columns, find errors, fix the clear ones, and save the clean data for the next notebooks.

Rules we follow:
- **Fixed here:** only clear errors that follow a fixed rule (empty rows, impossible values).
- **Left to the pipeline:** anything learned from the data (imputation, outliers, scaling, encoding). It must be fitted on train only, to avoid leakage.
- `spend_next_90d` is never used as a feature.
- A feature may only use the current row, the training data and `clientes.csv`.

Each section ends with a **Key insight**: what we found, and what to do in preprocessing.

**Next notebooks:** `02_eda.ipynb` explores the clean data saved at the end of this notebook (Section 8). `03_feature_engineering_selection.ipynb` builds and selects the features.

## 1. Loading the data and understanding the columns

In [50]:
import sys
import importlib
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

import src.data as data_module
importlib.reload(data_module)

from src.data import (
    check_prior_transactions,
    flag_prior_maximum,
    mark_prior_maximum_missing,
)

In [51]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import src.data as rolling
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import spearmanr
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = "../data"

In [52]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
clientes = pd.read_csv(f"{DATA_DIR}/clientes.csv")

print("train:   ", train.shape)
print("test:    ", test.shape)
print("clientes:", clientes.shape)

train:    (6534, 40)
test:     (986, 38)
clientes: (1143, 13)


In [53]:
# Which columns are in train but not in test (should be exactly the two targets)
print("Only in train:", sorted(set(train.columns) - set(test.columns)))
print("Only in test: ", sorted(set(test.columns) - set(train.columns)))

Only in train: ['repeat_purchase_90d', 'spend_next_90d']
Only in test:  []


In [54]:
train.head()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,spend_next_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,1,359.04,79.0,126.0,2.0,858.30,429.150000,112.656252,349.49,508.81,47.000000,NaN,0.0,0.00,1.0,349.49,2.0,858.30,2.0,858.30,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,10000000,-0.660448
1,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,1,592.71,244.0,616.0,5.0,6722.83,1344.566000,1683.795178,84.38,4222.15,93.000000,117.657696,0.0,0.00,0.0,0.00,0.0,0.00,4.0,2500.68,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,10000001,0.924510
2,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,1,423.72,9.0,610.0,90.0,17590.08,195.445333,190.389932,4.54,1002.15,6.752809,6.384073,2.0,111.27,9.0,1928.55,28.0,5290.25,49.0,7899.80,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,10000002,-0.258730
3,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,0,NaN,97.0,97.0,1.0,270.33,270.330000,NaN,270.33,270.33,NaN,NaN,0.0,0.00,0.0,0.00,1.0,270.33,1.0,270.33,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,10000003,1.448402
4,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,1,455.73,85.0,1575.0,25.0,12803.02,512.120800,406.696963,22.58,1736.74,62.083333,66.246520,0.0,0.00,2.0,184.13,4.0,2062.50,4.0,2062.50,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,10000004,-1.058890


### The columns

The brief groups the columns as below (Portuguese names translated). We save these groups in `column_groups` to use them later.

| Group | Columns | Meaning |
|---|---|---|
| Identifiers | `ID`, `customer_id` | Row ID; customer key to join `clientes.csv` |
| Targets | `repeat_purchase_90d`, `spend_next_90d` | Buys again within 90 days (0/1); how much they spend in those 90 days (EUR). Train only |
| Date | `purchase_date` | Anonymised date; order and intervals preserved |
| Current purchase: money | `Total a Pagar` (total to pay), `Total Bruto` (gross), `Total Liquido` (net), `Total Imp_` (tax), `Total Portes` (shipping), `Total Desc_ Global` (global discount), `Total Desc_ Linha` (line discounts), `Custo` (cost) | Values of this invoice |
| Current purchase: size | `N_ Detalhes` (number of invoice lines), `invoice_count_current_purchase` | |
| Current purchase: categories | `payment_type`, `salesperson`, `commercial_zone`, `transport_method`, `warehouse`, `document_series` | |
| History: timing | `days_since_previous_purchase`, `customer_tenure_days`, `prior_purchase_count`, `prior_mean_gap_days`, `prior_std_gap_days` | Recency, time as a customer, number of past purchases, gaps between past purchases |
| History: value | `prior_total_spend`, `prior_mean/std/min/max_purchase_value` | Summary of past purchase values |
| Recent windows: count | `purchase_count_30d/90d/180d/365d` | Purchases in the previous N days |
| Recent windows: spend | `spend_30d/90d/180d/365d` | Spend in the previous N days |
| Noise | `random_noise` | To check if it is useful |

In [55]:
column_groups = {
    "identifiers": ["ID", "customer_id"],
    "targets": ["repeat_purchase_90d", "spend_next_90d"],
    "date": ["purchase_date"],
    "current_monetary": ["Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_",
                         "Total Portes", "Total Desc_ Global", "Total Desc_ Linha", "Custo"],
    "current_details": ["N_ Detalhes", "invoice_count_current_purchase"],
    "categoricals": ["payment_type", "salesperson", "commercial_zone",
                     "transport_method", "warehouse", "document_series"],
    "hist_timing": ["days_since_previous_purchase", "customer_tenure_days", "prior_purchase_count",
                    "prior_mean_gap_days", "prior_std_gap_days"],
    "hist_value": ["prior_total_spend", "prior_mean_purchase_value", "prior_std_purchase_value",
                   "prior_min_purchase_value", "prior_max_purchase_value"],
    "rolling_count": ["purchase_count_30d", "purchase_count_90d", "purchase_count_180d", "purchase_count_365d"],
    "rolling_spend": ["spend_30d", "spend_90d", "spend_180d", "spend_365d"],
    "noise": ["random_noise"],
}

# Sanity check: every train column belongs to exactly one group
grouped = [c for cols in column_groups.values() for c in cols]
print("Columns in train but not in any group:", set(train.columns) - set(grouped))
print("Columns in a group but not in train:  ", set(grouped) - set(train.columns))
print("Columns listed twice:                 ", len(grouped) - len(set(grouped)))

Columns in train but not in any group: set()
Columns in a group but not in train:   set()
Columns listed twice:                  0


### Data types and missing values

One row per column: type in train and test, missing values, and number of distinct values. A column that is `int` in test but `float` in train usually has missing values in train (pandas turns integers with NaN into floats).

In [56]:
overview = pd.DataFrame({
    "dtype_train": train.dtypes,
    "dtype_test": test.dtypes.reindex(train.columns),
    "missing_train": train.isna().sum(),
    "missing_test": test.isna().sum().reindex(train.columns),
    "missing_%_train": (train.isna().mean() * 100).round(1),
    "missing_%_test": (test.isna().mean().reindex(train.columns) * 100).round(1),
    "n_unique_train": train.nunique(),
})
overview

,dtype_train,dtype_test,missing_train,missing_test,missing_%_train,missing_%_test,n_unique_train
Total a Pagar,float64,float64,65,0.0,1.0,0.0,6122
Total Bruto,float64,float64,65,0.0,1.0,0.0,6038
Total Liquido,float64,float64,65,0.0,1.0,0.0,6093
Total Imp_,float64,float64,65,0.0,1.0,0.0,5406
Total Portes,float64,float64,65,0.0,1.0,0.0,215
Total Desc_ Global,float64,float64,65,0.0,1.0,0.0,109
Total Desc_ Linha,float64,float64,65,0.0,1.0,0.0,3519
Custo,float64,float64,65,0.0,1.0,0.0,5736
N_ Detalhes,float64,int64,65,0.0,1.0,0.0,171
invoice_count_current_purchase,float64,int64,65,0.0,1.0,0.0,5


In [57]:
# Columns whose dtype differs between train and test
overview[overview["dtype_train"] != overview["dtype_test"]].dropna(subset=["dtype_test"])

,dtype_train,dtype_test,missing_train,missing_test,missing_%_train,missing_%_test,n_unique_train
N_ Detalhes,float64,int64,65,0.0,1.0,0.0,171
invoice_count_current_purchase,float64,int64,65,0.0,1.0,0.0,5
customer_tenure_days,float64,int64,65,0.0,1.0,0.0,2121
prior_purchase_count,float64,int64,65,0.0,1.0,0.0,277
purchase_count_30d,float64,int64,65,0.0,1.0,0.0,12
purchase_count_90d,float64,int64,65,0.0,1.0,0.0,23
purchase_count_180d,float64,int64,65,0.0,1.0,0.0,38
purchase_count_365d,float64,int64,65,0.0,1.0,0.0,61


### clientes.csv

Extra table with one row per customer: location (anonymised) and the year they became a customer.

In [58]:
clientes.head()

,customer_id,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
0,842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,City CY417: Missing Cell Port,NaN,Postal code PC828: #REF! Delivery Route,NaN,Address AD1120: Cell Without a Home,NaN,Excel Is Not A Database Ltd 150,2018
1,770941,Country CT9: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,Postal code PC56: #REF! Delivery Route,NaN,Address AD62: Cell Without a Home,NaN,Excel Is Not A Database Ltd 1064,2021
2,827053,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC827: #REF! Delivery Route,Street ST170: PowerPoint New Walk,Address AD449: Cell Without a Home,NaN,Excel Is Not A Database Ltd 113,2016
3,494648,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD454: Cell Without a Home,NaN,Excel Is Not A Database Ltd 430,2015
4,429291,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD453: Cell Without a Home,NaN,Excel Is Not A Database Ltd 673,2015


In [59]:
pd.DataFrame({
    "dtype": clientes.dtypes,
    "missing": clientes.isna().sum(),
    "missing_%": (clientes.isna().mean() * 100).round(1),
    "n_unique": clientes.nunique(),
})

,dtype,missing,missing_%,n_unique
customer_id,int64,0,0.0,1143
country_or_region,str,0,0.0,11
commercial_region,str,114,10.0,37
island,str,578,50.6,10
municipality,str,159,13.9,183
city,str,546,47.8,417
postal_area,str,71,6.2,271
postal_code,str,170,14.9,828
street,str,598,52.3,511
address,str,6,0.5,1120


### Key insight - Section 1

**Found**
- train and test have the same 38 predictors; train also has the two targets.
- In train, 65 values are missing in almost every column, always on the same rows (checked in Section 2). test has none.
- Those rows are why some integer columns show as `float` in train.
- `clientes.csv` has one row per customer. Most columns are anonymised addresses with many missing values, and `customer_type` is completely empty.

**For preprocessing**
- Nothing to fix in the column types: they match once the empty rows are removed (Section 2).
- From `clientes.csv`, drop `customer_type` and keep only the few useful columns (notebook 02, Section 7).

In [60]:
# Plot style used in every chart below
pio.templates["eda"] = go.layout.Template(layout=dict(
    font=dict(size=13), title=dict(font=dict(size=17)), height=450,
    margin=dict(t=80, l=60, r=30, b=50), hoverlabel=dict(bgcolor="white"),
    uniformtext=dict(minsize=11, mode="show")))
pio.templates.default = "plotly_white+eda"

BLUE, ORANGE, GREY, RED = "#2f6690", "#d9822b", "#9aa5b1", "#c0392b"
TARGET, SPEND = "repeat_purchase_90d", "spend_next_90d"

# Working copy: every fix below is applied to `df`; `train` stays untouched for comparison
df = train.copy()
df["purchase_date"] = pd.to_datetime(df["purchase_date"])
test["purchase_date"] = pd.to_datetime(test["purchase_date"])


def rate_by_bins(data, column, bins, labels=None, fig=None, row=None, col=None, title=None, xlabel=None):
    """Bar chart of the repeat-purchase rate per bin of `column`, with the row count on each bar.
    Draws into subplot (row, col) of `fig` when given, otherwise shows its own figure."""
    groups = pd.cut(data[column], bins=bins, labels=labels)
    summary = data.groupby(groups, observed=True)[TARGET].agg(["mean", "size"])
    own_figure = fig is None
    fig = go.Figure() if own_figure else fig
    pos = {} if own_figure else dict(row=row, col=col)
    overall = data[TARGET].mean()
    fig.add_bar(x=summary.index.astype(str), y=summary["mean"], marker_color=BLUE, showlegend=False,
                text=[f"n={n}" for n in summary["size"]], textposition="outside",
                hovertemplate="%{x}<br>repeat rate: %{y:.2f}<br>%{text}<extra></extra>", **pos)
    fig.add_hline(y=overall, line_dash="dash", line_color=ORANGE, **pos)
    fig.update_yaxes(range=[0, 1.1], title_text="repeat rate", **pos)
    fig.update_xaxes(title_text=xlabel or column, **pos)
    if own_figure:
        fig.update_layout(title=f"{title or f'Repeat rate by {column}'}<br><sup>dashed line = overall rate ({overall:.2f})</sup>")
        fig.show()
    return summary


def add_density(fig, values, color, name, row=1, col=1, showlegend=True):
    """Smoothed distribution (KDE) of `values` as a filled line in subplot (row, col)."""
    from scipy.stats import gaussian_kde
    values = values[np.isfinite(values)]
    grid = np.linspace(values.min(), values.max(), 200)
    r, g, b = (int(color[i:i + 2], 16) for i in (1, 3, 5))
    fig.add_scatter(x=grid, y=gaussian_kde(values)(grid), mode="lines", fill="tozeroy", name=name,
                    line=dict(color=color, width=2), fillcolor=f"rgba({r},{g},{b},0.3)",
                    legendgroup=name, showlegend=showlegend, row=row, col=col)

## 2. Empty rows and duplicates

In [61]:
feature_cols = [c for c in train.columns if c not in column_groups["identifiers"] + column_groups["targets"]]
empty_rows = df[feature_cols].isna().all(axis=1)
print("Rows with every predictor missing:", empty_rows.sum())
print("Columns that are still filled in those rows:", df.loc[empty_rows].columns[df.loc[empty_rows].notna().any()].tolist())
print("Repeat rate in those rows:", round(df.loc[empty_rows, TARGET].mean(), 3))

Rows with every predictor missing: 65
Columns that are still filled in those rows: ['repeat_purchase_90d', 'customer_id', 'ID']
Repeat rate in those rows: 0.508


In [62]:
missing_before = train.isna().sum()
missing_after = train.loc[~empty_rows].isna().sum()
compare = pd.DataFrame({"with empty rows": missing_before, "without empty rows": missing_after})
compare = compare[compare["with empty rows"] > 0].sort_values("with empty rows")

fig = go.Figure()
fig.add_bar(y=compare.index, x=compare["with empty rows"], orientation="h", name="with empty rows", marker_color=GREY)
fig.add_bar(y=compare.index, x=compare["without empty rows"], orientation="h", name="without empty rows", marker_color=BLUE)
fig.update_layout(title="Missing values per column, before and after removing the 65 empty rows",
                  xaxis_title="number of missing values", barmode="group", height=750)
fig.show()

In [63]:
# Exact duplicates, ignoring the ID (which is unique by construction)
dup_all = df.drop(columns="ID").duplicated()
print("Duplicates ignoring ID (all rows):     ", dup_all.sum())
print("  ...of which are empty rows:          ", (dup_all & empty_rows).sum())

df = df.loc[~empty_rows].copy()
print("Duplicates ignoring ID (after removal):", df.drop(columns="ID").duplicated().sum())
print("Same customer, same date:              ", df.duplicated(subset=["customer_id", "purchase_date"]).sum())
print("Rows left:", len(df))

Duplicates ignoring ID (all rows):      10
  ...of which are empty rows:           10
Duplicates ignoring ID (after removal): 0
Same customer, same date:               0
Rows left: 6469


### Key insight - Section 2

**Found**
- 65 rows are empty: only `ID`, `customer_id` and the target are filled, so there is nothing to learn from them.
- The only duplicates (10) are among these empty rows. Once they are removed there are **no duplicates**, not even the same customer on the same day.
- So each row is one distinct purchase.

**For preprocessing**
- Drop the 65 empty rows from train (done here). test has none, so no prediction is lost.
- No deduplication step is needed.

## 3. Impossible values and sentinel codes

Some columns must follow rules by definition, so a value that breaks a rule is wrong:
- mean purchase value = total spend / number of purchases
- min <= mean <= max <= total spend
- a longer window contains a shorter one: `spend_30d` <= `spend_90d` <= `spend_180d` <= `spend_365d` <= total spend (same for counts)
- a number of days cannot be negative

The table shows the 10 columns whose maximum is furthest above the 99th percentile (`max / p99`), and how many negative values each has.

In [64]:
num_cols = df.select_dtypes("number").columns.drop(column_groups["identifiers"] + column_groups["targets"])
summary = df[num_cols].describe(percentiles=[0.01, 0.5, 0.99]).T
summary["max / p99"] = summary["max"] / summary["99%"]
summary["n_negative"] = (df[num_cols] < 0).sum()
summary.sort_values("max / p99", ascending=False).round(2).head(10)

,count,mean,std,min,1%,50%,99%,max,max / p99,n_negative
prior_mean_purchase_value,5672.0,2821825.32,53041566.66,-2478.66,42.35,724.90,5230.09,1.000000e+09,191201.24,16
spend_365d,6274.0,2396534.88,48841133.46,-159115.18,0.00,2426.60,40876.80,1.000000e+09,24463.76,15
Total Desc_ Global,6469.0,0.35,5.12,0.00,0.00,0.00,7.85,2.184900e+02,27.85,0
prior_max_purchase_value,5866.0,2946.38,5901.55,3.31,75.46,1783.24,14433.41,2.675262e+05,18.54,0
Total Portes,6469.0,1.15,5.84,0.00,0.00,0.00,11.80,1.810600e+02,15.34,0
prior_mean_gap_days,5226.0,95.88,122.59,-4.00,5.79,68.44,514.10,3.199000e+03,6.22,16
prior_std_gap_days,5056.0,74.14,101.82,0.00,4.87,50.64,486.34,2.212540e+03,4.55,0
spend_30d,6469.0,457.14,1191.23,0.00,0.00,0.00,5396.49,2.229721e+04,4.13,0
days_since_previous_purchase,5673.0,101.53,165.70,-4.00,1.00,57.00,787.12,3.199000e+03,4.06,16
Total Imp_,6469.0,152.22,225.26,0.00,0.00,79.74,1026.79,4.169400e+03,4.06,0


### Distribution of each numeric variable

Pick a variable in the dropdown (top right).
- **Top:** histogram of the middle 98% of values, to see the shape.
- **Bottom:** box plot of **all** values; points far from the box are extreme values (e.g. the `1e9` placeholders).

Hover for exact numbers, drag to zoom, double-click to reset.

In [65]:
fig = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3], vertical_spacing=0.15,
                    subplot_titles=("Histogram of the middle 98% (1%-99%)", "Box plot of all values"))
buttons = []
for i, variable in enumerate(num_cols):
    values = df[variable].dropna()
    low, high = values.quantile([0.01, 0.99])
    bulk = values[values.between(low, high)]
    fig.add_histogram(x=bulk, nbinsx=50, marker_color=BLUE, name=variable, visible=(i == 0), row=1, col=1)
    fig.add_box(x=values, marker_color=BLUE, boxpoints="outliers", name="", visible=(i == 0), row=2, col=1)

    title = (f"{variable}<br><sup>median {values.median():,.2f} | max {values.max():,.2f} | "
             f"negatives {(values < 0).sum()} | missing {df[variable].isna().sum()} | "
             f"outside 1%-99%: {len(values) - len(bulk)}</sup>")
    visible = [False] * (2 * len(num_cols))
    visible[2 * i] = visible[2 * i + 1] = True
    buttons.append(dict(label=variable, method="update", args=[{"visible": visible}, {"title.text": title}]))

fig.update_layout(title=buttons[0]["args"][1]["title.text"], height=600, showlegend=False,
                  updatemenus=[dict(buttons=buttons, x=1, xanchor="right", y=1.2, yanchor="top")])
fig.update_yaxes(title_text="number of rows", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.show()

Two columns stand out: `prior_mean_purchase_value` and `spend_365d` reach exactly **1,000,000,000** and have negative values. `1e9` is a placeholder (a *sentinel*), not a real amount.

### 3.1 prior_purchase_count

We compared `prior_purchase_count` with the number of usable transactions found for the same customer **before each purchase date**. The status summary indicates whether the counts match, fewer or more transactions were found, or the comparison could not be performed.


In [66]:
history_check = check_prior_transactions(df, history=train)

# Summary of the results
display(history_check["status"].value_counts())


status
count_matches    5340
fewer_found      1129
Name: count, dtype: int64

We will investigate customers with fewer previous transactions found than reported in `prior_purchase_count`. We will count the unique affected customers and check whether they have records in the original training file with all transaction predictors missing, listing the corresponding transaction IDs. This will help assess whether incomplete records could explain the gaps in the available purchase history.

In [67]:
fewer_found = history_check.loc[history_check["status"].eq("fewer_found")]
print("Unique customers with fewer transactions found:",
      fewer_found["customer_id"].nunique())


cols_to_check = train.columns.difference([
    "ID", "customer_id",
    "repeat_purchase_90d", "spend_next_90d",
])
empty_original = train.loc[
    train[cols_to_check].isna().all(axis=1)
]

# Customers with fewer earlier transactions than expected
affected_ids = history_check.loc[
    history_check["status"].eq("fewer_found"), "customer_id"
].unique()

# List the empty transaction IDs belonging to each customer
empty_ids_by_customer = (
    empty_original.groupby("customer_id")["ID"].agg(list)
)

customer_check = pd.DataFrame({"customer_id": affected_ids})
customer_check["empty_row_IDs"] = (
    customer_check["customer_id"].map(empty_ids_by_customer)
)
customer_check["has_empty_row"] = customer_check["empty_row_IDs"].notna()

print("Original empty rows:", len(empty_original))
print("Affected customers:", len(customer_check))
print("Affected customers with an empty row:",
      customer_check["has_empty_row"].sum())

display(customer_check.head())

Unique customers with fewer transactions found: 46
Original empty rows: 65
Affected customers: 46
Affected customers with an empty row: 46


,customer_id,empty_row_IDs,has_empty_row
0,122356,[10002462],True
1,821421,[10004163],True
2,239651,"[10004187, 10005141]",True
3,248877,[10002499],True
4,258332,"[10002381, 10005269, 10006618]",True


The check identified **65 empty training rows** and confirmed that **all 46 customers with incomplete purchase histories have at least one of these records**. The empty records retain customer and transaction IDs but lack purchase dates, amounts and other transaction details.

These unavailable records may explain why the reported historical counts and totals exceed what we can observe. One unavailable purchase can affect several later rows. However, because the empty records lack dates, we cannot identify exactly which record occurred before each purchase.

**We checked history completeness to decide which values we could safely reconstruct.** A missing purchase could change the historical maximum, while a gap calculated from visible dates could be too large. Therefore, we could not automatically treat the available transactions as the complete history.

**Treatment:**

- Exclude the 65 empty rows from model training while keeping the original file for reference.
- Keep the affected customers’ other usable purchases and valid historical information. Do not reduce reported counts to the number of available records.
- Reconstruct values only when verified using transaction history, other reliable columns or independent checks. Otherwise, leave the invalid or missing field as `NaN` for imputation fitted on training data.
- For test repairs, use only the row’s own information and earlier training transactions.

### 3.2 `prior_mean_purchase_value`

In [68]:
has_history = df["prior_purchase_count"] > 0
recomputed_mean = df["prior_total_spend"] / df["prior_purchase_count"]
recomputed_mean = recomputed_mean.where(has_history)
stored_mean = df["prior_mean_purchase_value"]

sentinel = stored_mean == 1e9
negative = stored_mean < 0
matches = (stored_mean - recomputed_mean).abs() <= 0.01
inflated = stored_mean.notna() & ~sentinel & ~negative & ~matches & has_history

print("Matches total / count:", matches.sum())
print("Sentinel 1e9:         ", sentinel.sum())
print("Negative:             ", negative.sum(), "-> equal to minus the true mean:",
      np.allclose(-stored_mean[negative], recomputed_mean[negative]))
print("Inflated:             ", inflated.sum(), "-> stored / true ratio:",
      (stored_mean[inflated] / recomputed_mean[inflated]).describe()[["min", "50%", "max"]].round(1).to_dict())
print("Missing with history: ", (stored_mean.isna() & has_history).sum())

Matches total / count: 5594
Sentinel 1e9:          16
Negative:              16 -> equal to minus the true mean: True
Inflated:              46 -> stored / true ratio: {'min': 8.5, '50%': 16.1, 'max': 24.8}
Missing with history:  194


In [69]:
plot_df = pd.DataFrame({"true": recomputed_mean, "stored": stored_mean}).dropna()
plot_df = plot_df[(plot_df["stored"] > 0) & (plot_df["stored"] < 1e9)]
is_inflated = inflated.loc[plot_df.index]

lims = [1, 3e5]
fig = go.Figure()
fig.add_scatter(x=plot_df.loc[~is_inflated, "true"], y=plot_df.loc[~is_inflated, "stored"], mode="markers",
                marker=dict(size=5, color=BLUE, opacity=0.4), name="consistent")
fig.add_scatter(x=plot_df.loc[is_inflated, "true"], y=plot_df.loc[is_inflated, "stored"], mode="markers",
                marker=dict(size=8, color=RED), name=f"inflated ({is_inflated.sum()})")
fig.add_scatter(x=lims, y=lims, mode="lines", line=dict(color=GREY, dash="dash", width=1), name="stored = true")
fig.update_xaxes(type="log", range=np.log10(lims), title_text="prior_total_spend / prior_purchase_count")
fig.update_yaxes(type="log", range=np.log10(lims), title_text="stored prior_mean_purchase_value")
fig.update_layout(title="Stored mean vs mean recomputed from total and count", width=700, height=600)
fig.show()

Almost every point lies on the diagonal (stored = true). The errors are:
- 16 placeholders (`1e9`)
- 16 sign flips (exactly minus the true value)
- 46 values 8 to 25 times too large

**Fix:** total spend and count are always available, so we recompute the mean as total / count. This fixes all 78 errors and also fills the 194 missing values.

In [70]:
df["prior_mean_purchase_value"] = recomputed_mean
check = df["prior_mean_purchase_value"]
print("Mean outside [min, max] after the fix:",
      ((check < df["prior_min_purchase_value"] - 0.01) | (check > df["prior_max_purchase_value"] + 0.01)).sum())

Mean outside [min, max] after the fix: 0


### 3.3 `prior_max_purchase_value`

We checked `prior_max_purchase_value` against the limits implied by the historical purchase count, total spend, minimum and standard deviation. A maximum is flagged when it falls below the calculated average or exceeds the smallest available upper bound, allowing **€0.01 for rounding differences**.

The number of flagged rows and the first five examples are displayed separately for training and test data. This check identifies inconsistencies for further review without modifying either dataset.

In [71]:
flagged_train = flag_prior_maximum(df)
flagged_test = flag_prior_maximum(test)

print(f"Train: {len(flagged_train)} rows flagged")
display(flagged_train.head())

print(f"Test: {len(flagged_test)} rows flagged")
display(flagged_test.head())

Train: 48 rows flagged


,ID,customer_id,prior_purchase_count,prior_total_spend,prior_min_purchase_value,prior_max_purchase_value,prior_std_purchase_value,recomputed_mean,upper_from_min,upper_from_std,upper_limit
263,10000300,928271,5.0,3456.32,87.48,22844.431540,519.423999,691.264000,3106.40,1730.111999,1730.111999
732,10000853,864223,25.0,40285.74,162.61,55316.611919,1399.492395,1611.429600,36383.10,8467.514132,8467.514132
940,10001102,738735,9.0,16658.36,56.33,55269.483555,1337.288793,1850.928889,16207.72,5633.352784,5633.352784
966,10001129,358947,5.0,2494.49,294.46,21941.361234,362.767370,498.898000,1316.65,1224.432740,1224.432740
1261,10001474,475032,21.0,15632.63,17.80,25095.641836,603.679923,744.410952,15276.63,3444.149640,3444.149640


Test: 7 rows flagged


,ID,customer_id,prior_purchase_count,prior_total_spend,prior_min_purchase_value,prior_max_purchase_value,prior_std_purchase_value,recomputed_mean,upper_from_min,upper_from_std,upper_limit
134,10001137,547470,21,14225.51,88.40,29727.285468,359.258253,677.405238,12457.51,2284.056988,2284.056988
185,10001535,441324,57,69434.25,15.45,80406.616803,1259.858223,1218.144737,68569.05,10646.060389,10646.060389
308,10002646,683049,17,65786.08,909.53,126446.295493,1917.824563,3869.769412,51233.60,11541.067666,11541.067666
334,10002801,429291,14,9267.45,8.27,42174.683300,443.986148,661.960714,9159.94,2262.775535,2262.775535
413,10003405,519184,104,40963.09,2.72,61162.314197,563.553869,393.875865,40682.93,6113.322973,6113.322973


In [72]:
df, max_audit_train = mark_prior_maximum_missing(
    df, history=train, flagged_ids=flagged_train["ID"]
)

test, max_audit_test = mark_prior_maximum_missing(
    test, history=train, flagged_ids=flagged_test["ID"]
)

for name, audit in [
    ("Train", max_audit_train),
    ("Test", max_audit_test),
]:
    print(name)
    display(audit["recovery_status"].value_counts())

Train


recovery_status
complete_history               36
incomplete_but_max_verified    12
Name: count, dtype: int64

Test


recovery_status
unresolved                     5
incomplete_but_max_verified    1
complete_history               1
Name: count, dtype: int64


**Fix:** rebuild the max from the client's earlier purchases in train (the biggest `Total a Pagar` before this one).
- If 1 or 2 earlier purchases are missing from train, their total is `prior_total_spend` minus what we see. If that amount is smaller than the max we see, it can't change the max.
- Anything that still can't be rebuilt becomes NaN and is imputed in the pipeline.

### 3.3.1 Rebuilt `prior_max_purchase_value`

In [73]:
from src.preprocessing import rebuild_max
df = rebuild_max(df, max_audit_train)
test = rebuild_max(test, max_audit_test)

In [74]:
for name, data, audit in [
    ("Train", df, max_audit_train),
    ("Test", test, max_audit_test),
]:
    expected = audit.set_index("ID")["verified_maximum"]
    actual = data.set_index("ID").loc[
        expected.index, "prior_max_purchase_value"
    ]

    matches = (
        (actual - expected).abs().le(0.01)
        | (actual.isna() & expected.isna())
    )

    print(f"{name}: {matches.sum()}/{len(audit)} audit checks passed")
    print(
        "Rebuilt:", actual.notna().sum(),
        "| Still missing:", actual.isna().sum(),
    )
    print(
        "Remaining inconsistent maxima:",
        len(flag_prior_maximum(data)),
    )
    print()

Train: 48/48 audit checks passed
Rebuilt: 48 | Still missing: 0
Remaining inconsistent maxima: 0

Test: 7/7 audit checks passed
Rebuilt: 2 | Still missing: 5
Remaining inconsistent maxima: 0



### 3.4 Rolling spending consistency checks


We will check the **30, 90, 180 and 365-day spending features** in both training and test data, using historical summaries and earlier training transactions as references.

Values identified as inconsistent will be replaced with `NaN`. Their original values and any verified replacements will be retained in separate audit tables for later reconstruction. We will then summarize the number of flagged values for each window and dataset.


The rolling spending features represent how much a customer spent during the **30, 90, 180 or 365 days before each purchase**, excluding the current purchase.

We checked these values in two ways:

- **Transaction comparison:** when the expected earlier purchases are available, we sum their amounts and compare the result with the reported spending.
- **Consistency rules:** we check whether the value respects historical limits. For example, spending during the previous year cannot exceed total spending across the customer’s entire previous history.



In [75]:

importlib.reload(rolling)

windows = (30, 90, 180, 365)

df, rolling_audit_train = rolling.mark_rolling_spend_missing(
    df, history=train, windows=windows
)

test, rolling_audit_test = rolling.mark_rolling_spend_missing(
    test, history=train, windows=windows
)

# Flags per window
summary = pd.DataFrame({
    "Train": rolling_audit_train["window_days"].value_counts(),
    "Test": rolling_audit_test["window_days"].value_counts(),
}).reindex(windows).fillna(0).astype(int)

summary.index.name = "window_days"
display(summary)



,Train,Test
window_days,,
30,0,0
90,0,0
180,0,0
365,70,11


 The checks flagged **70 training values and 11 test values**, all in `spend_365d`. No values were flagged in the shorter windows.

The flagged values were replaced with `NaN`. Their original values and any verified replacement amounts were retained in the audit tables for later reconstruction. Incomplete transaction history alone was not treated as evidence of an incorrect value.

In [76]:
for name, audit in [
    ("Train", rolling_audit_train),
    ("Test", rolling_audit_test),
]:
    print(name)
    display(
        audit.groupby(["window_days", "recovery_status"])
        .size()
        .unstack(fill_value=0)
    )

Train


recovery_status,complete_history,unresolved
window_days,,
365,65,5


Test


recovery_status,unresolved,verified_from_row
window_days,,
365,10,1


In [77]:
from src.preprocessing import rebuild_rolling_spend


df = rebuild_rolling_spend(df, rolling_audit_train)
test = rebuild_rolling_spend(test, rolling_audit_test)

In [78]:
checks = []

for name, data, audit in [
    ("Train", df, rolling_audit_train),
    ("Test", test, rolling_audit_test),
]:
    for column, group in audit.groupby("spend_column"):
        expected = group.set_index("ID")["verified_spend"]
        actual = data.set_index("ID").loc[expected.index, column]

        matches = (
            (actual - expected).abs().le(0.01)
            | (actual.isna() & expected.isna())
        )

        checks.append({
            "Dataset": name,
            "Column": column,
            "Rebuilt": actual.notna().sum(),
            "Still missing": actual.isna().sum(),
            "Checks passed": matches.all(),
        })

display(pd.DataFrame(checks))

,Dataset,Column,Rebuilt,Still missing,Checks passed
0,Train,spend_365d,65,5,True
1,Test,spend_365d,1,10,True


In [79]:
windows = ["30d", "90d", "180d", "365d"]
rows = []
for short, long in zip(windows[:-1], windows[1:]):
    rows.append({"rule": f"count_{short} <= count_{long}",
                 "violations": (df[f"purchase_count_{short}"] > df[f"purchase_count_{long}"]).sum()})
    rows.append({"rule": f"spend_{short} <= spend_{long}",
                 "violations": (df[f"spend_{short}"] > df[f"spend_{long}"] + 0.01).sum()})
rows.append({"rule": "count_365d <= prior_purchase_count",
             "violations": (df["purchase_count_365d"] > df["prior_purchase_count"]).sum()})
rows.append({"rule": "spend_365d <= prior_total_spend",
             "violations": (df["spend_365d"] > df["prior_total_spend"] + 0.01).sum()})
pd.DataFrame(rows)

,rule,violations
0,count_30d <= count_90d,0
1,spend_30d <= spend_90d,0
2,count_90d <= count_180d,0
3,spend_90d <= spend_180d,0
4,count_180d <= count_365d,0
5,spend_180d <= spend_365d,0
6,count_365d <= prior_purchase_count,0
7,spend_365d <= prior_total_spend,0


In [80]:
valid = df[(df["spend_365d"] >= 0) & (df["spend_365d"] < 1e9)]
too_big = valid["spend_365d"] > valid["prior_total_spend"] + 0.01

fig = go.Figure()
fig.add_scatter(x=valid.loc[~too_big, "prior_total_spend"] + 1, y=valid.loc[~too_big, "spend_365d"] + 1, mode="markers",
                marker=dict(size=5, color=BLUE, opacity=0.4), name="valid")
fig.add_scatter(x=valid.loc[too_big, "prior_total_spend"] + 1, y=valid.loc[too_big, "spend_365d"] + 1, mode="markers",
                marker=dict(size=8, color=RED), name=f"spend_365d > total ({too_big.sum()})")
fig.add_scatter(x=[1, 1e6], y=[1, 1e6], mode="lines", line=dict(color=GREY, dash="dash", width=1), name="equal")
fig.update_xaxes(type="log", title_text="prior_total_spend + 1")
fig.update_yaxes(type="log", title_text="spend_365d + 1")
fig.update_layout(title="Spend in the last year cannot exceed all-time spend", width=700, height=600)
fig.show()

### 3.5 Negative day counts

In [81]:
gap_cols = ["days_since_previous_purchase", "prior_mean_gap_days"]
fig = make_subplots(rows=1, cols=2, subplot_titles=[f"{col} (values <= 15)" for col in gap_cols])
for i, col in enumerate(gap_cols, start=1):
    counts = df[col].clip(upper=15).round().value_counts().sort_index()
    fig.add_bar(x=counts.index, y=counts.values, marker_color=[RED if v < 0 else BLUE for v in counts.index],
                showlegend=False, hovertemplate="%{x} days: %{y} rows<extra></extra>", row=1, col=i)
    fig.update_xaxes(title_text="days (15 = 15 or more)", row=1, col=i)
fig.update_layout(height=420)
fig.show()

for col in gap_cols:
    print(col, "negative values:", df.loc[df[col] < 0, col].value_counts().to_dict())

days_since_previous_purchase negative values: {-4.0: 16}
prior_mean_gap_days negative values: {-4.0: 16}


In [82]:
# Is -4 a real value? Compare with the gap to the customer's previous purchase in train
ordered = df.sort_values(["customer_id", "purchase_date"])
gap_in_train = ordered.groupby("customer_id")["purchase_date"].diff().dt.days
compare_gap = pd.DataFrame({"stored": ordered["days_since_previous_purchase"], "gap_in_train": gap_in_train}).dropna()

print("stored == gap computed from dates:", (compare_gap["stored"] == compare_gap["gap_in_train"]).mean().round(3))
print("gap in train for the rows stored as -4:", compare_gap.loc[compare_gap["stored"] < 0, "gap_in_train"].tolist())

# When they disagree (ignoring the -4), which one is bigger?
mismatch = compare_gap[(compare_gap["stored"] >= 0) & (compare_gap["stored"] != compare_gap["gap_in_train"])]
print("rows where they disagree:", len(mismatch),
      "| gap from dates is bigger:", (mismatch["gap_in_train"] > mismatch["stored"]).sum(),
      "| gap from dates is smaller:", (mismatch["gap_in_train"] < mismatch["stored"]).sum())

stored == gap computed from dates: 0.988
gap in train for the rows stored as -4: [3.0, 231.0, 350.0, 13.0, 77.0, 25.0, 21.0, 14.0, 112.0, 47.0, 279.0, 32.0, 63.0, 35.0, 111.0, 105.0]
rows where they disagree: 51 | gap from dates is bigger: 51 | gap from dates is smaller: 0


Before repair, the stored gap matched the gap between visible purchase dates in **98.8% of comparable rows**. In the 51 nonnegative disagreements, the calculated gap was always larger, consistent with missing purchases. This agreement does not prove that every customer’s history is complete.

Both day-gap fields contained **16 invalid `-4` values in training and 2 in test**.

We will first calculate the invalid values using other valid columns in the same row: customer tenure, previous purchase count and the other gap value.

When that is insufficient, we will use earlier training purchase dates only when the available purchase count and total spending match the reported history.

The next cell will apply these repairs to training and test data and record the changes in audit tables. Values that cannot be verified will become `NaN`.

In [83]:
from src.preprocessing import repair_negative_days

df, days_audit_train = repair_negative_days(df, history=train)
test, days_audit_test = repair_negative_days(test, history=train)

for name, audit in [
    ("Train", days_audit_train),
    ("Test", days_audit_test),
]:
    print(name)
    display(pd.crosstab(audit["column"], audit["recovery_status"]))

Train


recovery_status,complete_history,verified_from_row
column,,
days_since_previous_purchase,1,15
prior_mean_gap_days,2,14


Test


recovery_status,verified_from_row
column,
days_since_previous_purchase,2
prior_mean_gap_days,2


We first calculated the correct gap using other columns in the same transaction row: customer tenure, previous purchase count and the other valid gap value.

For example, if the first purchase was **100 days ago**, there were **3 previous purchases**, and their average gap was **30 days**, those purchases span `2 × 30 = 60` days. The last purchase was therefore **40 days ago**.

When these columns did not provide enough information, we used the customer’s earlier training purchase dates. We only used those dates when the available purchases matched **both the reported purchase count and total spending**, to check that the expected history was complete.

### 3.6 Missing days since the previous purchase

We identified rows where the customer had previous purchases (`prior_purchase_count > 0`) but `days_since_previous_purchase` was missing. Rows with no previous purchases were excluded because missing recency is expected there.

The check found **193 training rows affecting 131 unique customers**, and **30 test rows**. We stored these cases separately and displayed the first five examples from each dataset for inspection.

This step identified the missing values without changing the data.

In [84]:
problem_rows = {}

columns = [
    "ID",
    "customer_id",
    "prior_purchase_count",
    "days_since_previous_purchase",
    "customer_tenure_days",
    "prior_mean_gap_days",
]

for name, data in [("Train", df), ("Test", test)]:
    problem = (
        data["prior_purchase_count"].gt(0)
        & data["days_since_previous_purchase"].isna()
    )

    problem_rows[name] = data.loc[problem, columns].copy()

    print(
        f"{name}: {problem.sum()} affected rows, "
        f"{problem_rows[name]['customer_id'].nunique()} unique customers"
    )
    display(problem_rows[name].head(5))

Train: 193 affected rows, 131 unique customers


,ID,customer_id,prior_purchase_count,days_since_previous_purchase,customer_tenure_days,prior_mean_gap_days
50,10000064,673819,8.0,NaN,1225.0,155.000000
153,10000180,359193,27.0,NaN,1049.0,37.153846
215,10000249,703834,42.0,NaN,1111.0,26.951220
218,10000252,531725,5.0,NaN,807.0,179.500000
219,10000253,957583,3.0,NaN,133.0,14.500000


Test: 30 affected rows, 24 unique customers


,ID,customer_id,prior_purchase_count,days_since_previous_purchase,customer_tenure_days,prior_mean_gap_days
4,10000025,610152,5,NaN,386,77.000000
48,10000432,660784,9,NaN,572,62.125000
50,10000437,939092,16,NaN,686,44.333333
78,10000654,475032,44,NaN,3423,79.093023
84,10000701,858609,9,NaN,2228,236.375000


In [85]:
from src.preprocessing import repair_missing_recency
df, recency_audit_train = repair_missing_recency(df, history=train)
test, recency_audit_test = repair_missing_recency(test, history=train)

for name, audit in [
    ("Train", recency_audit_train),
    ("Test", recency_audit_test),
]:
    print(name)
    display(audit["recovery_status"].value_counts())



Train


recovery_status
verified_from_row    188
complete_history       5
Name: count, dtype: int64

Test


recovery_status
verified_from_row    30
Name: count, dtype: int64

We filled missing `days_since_previous_purchase` values using information in other columns: the time since the customer’s first purchase, the number of previous purchases and the average time between them.

When this information was insufficient, we used earlier training purchase dates after checking that the purchase count and total spending matched the reported history.

- **Train:** 193 values were filled—188 using other columns and 5 using complete purchase history.
- **Test:** all 30 values were filled using other columns.

Rows with no previous purchases kept `NaN`.

## 3.7 `prior_mean_gap_days`

In [86]:
for name, data in [("Train", df), ("Test", test)]:
    problem = (
        data["prior_mean_gap_days"].isna()
        & data["prior_purchase_count"].ge(2)
    )

    print(f"{name}: {problem.sum()} missing mean gaps")
    display(data.loc[problem, [
        "ID", "prior_purchase_count", "prior_mean_gap_days"
    ]].head())

Train: 195 missing mean gaps


,ID,prior_purchase_count,prior_mean_gap_days
12,10000015,23.0,NaN
21,10000028,49.0,NaN
24,10000031,6.0,NaN
65,10000079,3.0,NaN
76,10000091,48.0,NaN


Test: 30 missing mean gaps


,ID,prior_purchase_count,prior_mean_gap_days
53,10000452,127,NaN
55,10000460,10,NaN
90,10000771,25,NaN
124,10001034,29,NaN
164,10001353,22,NaN


In [87]:
from src.preprocessing import repair_missing_mean_gap

df, mean_gap_audit_train = repair_missing_mean_gap(df)
test, mean_gap_audit_test = repair_missing_mean_gap(test)

for name, audit in [
    ("Train", mean_gap_audit_train),
    ("Test", mean_gap_audit_test),
]:
    print(
        f"{name}: {audit['verified_value'].notna().sum()} filled; "
        f"{audit['verified_value'].isna().sum()} unresolved"
    )

Train: 195 filled; 0 unresolved
Test: 30 filled; 0 unresolved


We filled missing values using the time since their first purchase, the time since their last purchase and their previous purchase count.

We restored **195 training values and 30 test values**, with none unresolved.

Rows with fewer than two previous purchases kept `NaN`, because there is no gap between previous purchases to average. The original and replacement values were recorded so each repair can be checked.

In [88]:
checks = []

for name, data in [("Train", df), ("Test", test)]:
    n = data["prior_purchase_count"]
    missing = data["prior_std_gap_days"].isna()

    checks.append({
        "Dataset": name,
        "Expected missing: fewer than 3 purchases": (
            missing & n.lt(3)
        ).sum(),
        "Unexpected missing: 3+ purchases": (
            missing & n.ge(3)
        ).sum(),
        "Unexpected present: fewer than 3 purchases": (
            ~missing & n.lt(3)
        ).sum(),
    })

display(pd.DataFrame(checks).set_index("Dataset"))

,Expected missing: fewer than 3 purchases,Unexpected missing: 3+ purchases,Unexpected present: fewer than 3 purchases
Dataset,,,
Train,1413,0,0
Test,69,0,0


In [89]:
for name, data in [("Train", df), ("Test", test)]:
    groups = pd.cut(
        data["prior_purchase_count"],
        bins=[-1, 0, 1, 2, float("inf")],
        labels=["0", "1", "2", "3+"],
    )

    print(name)
    display(
        data.groupby(groups, observed=False)["prior_std_purchase_value"]
        .agg(
            total_rows="size",
            missing=lambda values: values.isna().sum(),
            present="count",
        )
    )

Train


,total_rows,missing,present
prior_purchase_count,,,
0,603,603,0
1,445,445,0
2,365,0,365
3+,5056,0,5056


Test


,total_rows,missing,present
prior_purchase_count,,,
0,22,22,0
1,24,24,0
2,23,0,23
3+,917,0,917


### Key insight — Section 3

| Column | Problem | Train affected | Test affected | Treatment |
|---|---|---:|---:|---|
| `prior_mean_purchase_value` | Sentinels, negative or inflated values, and missing means with purchase history | 78 errors + 194 missing | 11 errors + 30 missing | Recompute as total spend / purchase count when count > 0. |
| `prior_max_purchase_value` | Inconsistent with the mean, total, minimum or standard deviation | 48 | 7 | Restore all 48 training values and 2 test values using verified replacements. Keep 5 test values as `NaN`. |
| `spend_365d` | Sentinels, negatives, spending-bound violations or transaction-sum mismatches | 70 | 11 | Restore 65 training values and 1 test value. Keep 5 training and 10 test values as `NaN`. |
| `days_since_previous_purchase` | Invalid `-4` values | 16 | 2 | Repair all using valid row summaries, with complete training history as a fallback. |
| `prior_mean_gap_days` | Invalid `-4` values | 16 | 2 | Repair all using valid row summaries, with complete training history as a fallback. |

All four spending windows were checked; only `spend_365d` had flagged inconsistencies. Existing missing day-gap values were preserved.

**For preprocessing**

- Apply the same cleaning rules to **training and test**, including the historical mean calculation.
- Repairs may use the row’s own summaries and **earlier training transactions**. Other test rows must not be used as history. During validation, use the fold’s training history.
- Impute remaining `NaN` values inside the pipeline, fitting the imputer on training data only.
- Evaluate missing-value indicators during validation to identify values filled by the imputer.

## 4. Consistency of the monetary columns

We will check that gross value minus discounts equals net value, and that net value plus tax equals the amount payable, using a **€0.02 tolerance**.

The checks will cover training and test data, reporting matches, non-matches and rows that cannot be checked. We will also test the equation with shipping added as a separate diagnostic.

In [90]:
tolerance = 0.02
rows = []

columns = [
    "Total Bruto", "Total Desc_ Linha", "Total Desc_ Global",
    "Total Liquido", "Total Imp_", "Total a Pagar", "Total Portes",
]

for dataset, data in [("Train", df), ("Test", test)]:
    amounts = (
        data[columns]
        .apply(pd.to_numeric, errors="coerce")
        .astype(float)
    )

    checks = {
        "Bruto - discounts = Liquido": (
            amounts["Total Bruto"]
            - amounts["Total Desc_ Linha"]
            - amounts["Total Desc_ Global"]
            - amounts["Total Liquido"]
        ),
        "Liquido + Imp = a Pagar": (
            amounts["Total Liquido"]
            + amounts["Total Imp_"]
            - amounts["Total a Pagar"]
        ),
        "Shipping diagnostic: Liquido + Imp + Portes = a Pagar": (
            amounts["Total Liquido"]
            + amounts["Total Imp_"]
            + amounts["Total Portes"]
            - amounts["Total a Pagar"]
        ),
    }

    for rule, diff in checks.items():
        usable = np.isfinite(diff)
        matched = diff.loc[usable].abs().le(tolerance)

        rows.append({
            "Dataset": dataset,
            "Check": rule,
            "Checked": usable.sum(),
            "Not checked": (~usable).sum(),
            "Matches": matched.sum(),
            "Non-matches": (~matched).sum(),
            "Match (%)": (
                round(100 * matched.mean(), 2)
                if len(matched) else np.nan
            ),
        })

display(pd.DataFrame(rows).set_index(["Dataset", "Check"]))

Checked  Not checked  Matches  Non-matches  Match (%)
Dataset Check                                                                                                    
Train   Bruto - discounts = Liquido                            6469            0     6469            0     100.00
        Liquido + Imp = a Pagar                                6469            0     6469            0     100.00
        Shipping diagnostic: Liquido + Imp + Portes = a...     6469            0     5416         1053      83.72
Test    Bruto - discounts = Liquido                             986            0      986            0     100.00
        Liquido + Imp = a Pagar                                 986            0      986            0     100.00
        Shipping diagnostic: Liquido + Imp + Portes = a...      986            0      917           69      93.00

### Key insight Section 4

**Found**

- Both main invoice equations matched **100% of checked training and test rows**, within €0.02: gross − discounts = net, and net + tax = amount payable.
- Adding shipping produced **1,053 training non-matches and 69 test non-matches**. Since net + tax already matched the payable amount, shipping did not need to be added separately to this equation.

**For preprocessing**

- No corrections were required by these invoice checks.
- Shipping diagnostic non-matches should not be treated as corrupted values.
- Tax and margin ratios, together with an indicator for zero cost, are explored in notebook 03.

## 5. Recovering the missing spend target

`repeat_purchase_90d = 1` means the customer made another purchase within the next 90 days. `spend_next_90d` represents the **total amount spent on all purchases during that period**, excluding the current purchase.

If spending is missing, we know the customer returned, but we do not know how much they spent.

We can calculate that total from later training purchases **only when the full 90-day period is covered and no purchases within it are missing**. Finding one later purchase is insufficient: its amount may represent only part of the customer’s spending.

In [91]:
# Purchases where the customer came back (repeat = 1) but the spend of the next 90 days is empty
missing_spend = (df[TARGET] == 1) & df[SPEND].isna()
print("Purchases with repeat = 1 but no spend:", missing_spend.sum(),
      "| from", df.loc[missing_spend, "customer_id"].nunique(), "customers")
df.loc[missing_spend, ["ID", "customer_id", "purchase_date", "Total a Pagar", TARGET, SPEND]].head(5)

Purchases with repeat = 1 but no spend: 196 | from 119 customers


,ID,customer_id,purchase_date,Total a Pagar,repeat_purchase_90d,spend_next_90d
5,10000005,821421,2023-07-15,640.78,1,NaN
42,10000055,769296,2021-01-17,1118.02,1,NaN
102,10000120,341652,2022-07-30,609.82,1,NaN
170,10000198,847373,2016-11-14,429.31,1,NaN
249,10000286,272951,2019-04-30,36.90,1,NaN


In [92]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

TARGET = "repeat_purchase_90d"
SPEND = "spend_next_90d"

from src.data import future_spend


# Keep the diagnostic separate from df.
observed_spend = pd.Series(np.nan, index=df.index, dtype=float)

ordered = df.sort_values(["customer_id", "purchase_date"])
for _, group in ordered.groupby("customer_id"):
    observed_spend.loc[group.index] = future_spend(group)


# Compare against supplied spending targets, allowing one cent.
known = df[SPEND].notna() & observed_spend.notna()

difference = (
    observed_spend.loc[known] - df.loc[known, SPEND]
).abs()

matches = difference.le(0.01)

print(
    f"Known targets matching within €0.01: "
    f"{matches.sum()} of {known.sum()} "
    f"({matches.mean():.1%})"
)


# Visual comparison: the dashed line represents equal values.
if known.any():
    upper = max(
        df.loc[known, SPEND].max(),
        observed_spend.loc[known].max()
    ) + 1

    fig = go.Figure()

    fig.add_scatter(
        x=df.loc[known, SPEND] + 1,
        y=observed_spend.loc[known] + 1,
        mode="markers",
        marker=dict(size=4, color="#2F6B91", opacity=0.4),
        name="Training purchases"
    )

    fig.add_scatter(
        x=[1, upper],
        y=[1, upper],
        mode="lines",
        line=dict(color="grey", dash="dash"),
        name="Equal values"
    )

    fig.update_xaxes(
        type="log",
        title_text="Supplied spend_next_90d (+1)"
    )
    fig.update_yaxes(
        type="log",
        title_text="Observed spending in the next 90 days (+1)"
    )
    fig.update_layout(
        title="Observed future spending versus the supplied target",
        width=700,
        height=600
    )
    fig.show()


# Check consistency between the two targets.
print(
    "Spending target present when repeat = 0:",
    (df[TARGET].eq(0) & df[SPEND].notna()).sum()
)

missing_spend = df[TARGET].eq(1) & df[SPEND].isna()

print("Missing spending targets when repeat = 1:", missing_spend.sum())

print(
    "Of these, cases with some future spending visible in train:",
    (missing_spend & observed_spend.gt(0)).sum()
)

# Ending before this date is necessary for coverage, but does not
# prove that every purchase within the window is available.
latest_training_date = df["purchase_date"].max()
window_end = df["purchase_date"] + pd.Timedelta(days=90)

print(
    "Of the missing targets, windows ending after the latest training date:",
    (missing_spend & window_end.gt(latest_training_date)).sum()
)



Known targets matching within €0.01: 3316 of 3425 (96.8%)


Spending target present when repeat = 0: 0
Missing spending targets when repeat = 1: 196
Of these, cases with some future spending visible in train: 176
Of the missing targets, windows ending after the latest training date: 54


### Key insight Section 5

**Found**

- The observed spending in the next 90 days matched **3,316 of 3,425 known targets (96.8%)**, within €0.01.
- Differences may reflect unavailable purchases or incomplete coverage of the 90-day period. Their causes have not been verified individually.
- Of **196 missing spending targets**, **176** had some future spending visible in train and **20** had none. These 176 sums still require completeness checks before recovery.
- **54 of the 196 cases** had a 90-day window ending after the latest training purchase date.

**For preprocessing**

- Leave missing targets unchanged until their full 90-day spending can be verified.
- Regression: use supplied targets and verified replacements; exclude rows whose spending target remains missing.
- Keep these rows for classification when their classification target and predictors are usable.
- Future purchases may be used to reconstruct the spending **target**, but must never become model **features**.

In [93]:
from src.preprocessing import recover_spending_targets
# Remove the old diagnostic column if it exists.
df = df.drop(columns="spend_rebuilt", errors="ignore")

# Training data only: these checks use future training purchases.
df, spend_recovery = recover_spending_targets(df)

print("Spending targets recovered:",
      spend_recovery["verified_spend"].notna().sum())

print("Returning purchases still without a spending target:",
      (df[TARGET].eq(1) & df[SPEND].isna()).sum())

display(spend_recovery["method"].value_counts())

Spending targets recovered: 135
Returning purchases still without a spending target: 61


method
verified_with_no_further_purchase    134
unresolved                            61
verified_with_later_purchase           1
Name: count, dtype: int64

We applied the recovery checks to rows where `repeat_purchase_90d = 1` but `spend_next_90d` was missing. We filled a target only when purchase counts, spending totals and coverage of the 90-day period could be verified.

| Recovery method | Targets recovered |
|---|---:|
| A later purchase confirmed the complete window | 1 |
| The last visible purchase’s no-return label confirmed the remaining period | 134 |
| **Total recovered** | **135** |

The remaining **61 targets** were left as `NaN`:**41** had positive but unverified spending sums because we could not confirm that the entire 90-day period was covered. There was no later training purchase at or beyond the window’s end, and the last visible purchase had `repeat_purchase_90d = 1`, indicating another purchase afterwards. Its date was unavailable, so we could not tell whether it fell inside the original 90-day window. The observed sum could therefore be incomplete, and **20** had no future spending visible in train. Existing nonmissing targets were preserved.

We recorded each attempted repair and its method separately in `spend_recovery`, and removed the old `spend_rebuilt` diagnostic column.

## 6. `random_noise`

In [94]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import mannwhitneyu, spearmanr

# Separate analysis data; df remains unchanged.
noise_data = (
    df[["random_noise", TARGET]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)
noise_data = noise_data.loc[noise_data[TARGET].isin([0, 1])]

# Transactions followed by a repeat purchase, with a known spending target.
returned = df.loc[df[TARGET].eq(1) & df[SPEND].notna()].copy()

noise_returned = (
    returned[["random_noise", SPEND]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
)

bins = [-np.inf, -1.5, -0.5, 0, 0.5, 1.5, np.inf]
labels = [
    "≤ -1.5", "(-1.5, -0.5]", "(-0.5, 0]",
    "(0, 0.5]", "(0.5, 1.5]", "> 1.5"
]

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "random_noise by class",
        "Repeat rate by bin",
        "Spending by bin (returners)"
    )
)

# 1. Compare the noise distributions between classes.
for value, color, label in [
    (0, GREY, "no repeat"),
    (1, BLUE, "repeat")
]:
    add_density(
        fig,
        noise_data.loc[noise_data[TARGET].eq(value), "random_noise"],
        color,
        label,
        row=1,
        col=1
    )

fig.update_xaxes(title_text="random_noise", row=1, col=1)
fig.update_yaxes(title_text="Density", row=1, col=1)

# 2. Look for changes in repeat rate across the noise range.
rate_by_bins(
    noise_data,
    "random_noise",
    bins,
    labels=labels,
    fig=fig,
    row=1,
    col=2
)

# 3. Look for changes in spending across the same bins.
noise_returned["noise_bin"] = pd.cut(
    noise_returned["random_noise"],
    bins=bins,
    labels=labels
)

fig.add_trace(
    go.Box(
        x=noise_returned["noise_bin"].astype(str),
        y=noise_returned[SPEND] + 1,
        marker_color=BLUE,
        boxpoints=False,
        showlegend=False
    ),
    row=1,
    col=3
)

fig.update_xaxes(
    title_text="random_noise bin",
    categoryorder="array",
    categoryarray=labels,
    tickangle=-35,
    row=1,
    col=3
)
fig.update_xaxes(tickangle=-35, row=1, col=2)
fig.update_yaxes(
    type="log",
    title_text="90-day spending + 1 (€)",
    row=1,
    col=3
)

fig.update_layout(
    title=(
        "Exploring random_noise and both targets"
        "<br><sup>Middle panel: dashed line = overall repeat rate "
        f"({noise_data[TARGET].mean():.1%})</sup>"
    ),
    width=1350,
    height=520,
    margin_t=110
)
fig.show()

# Exploratory row-level tests: customers appear in multiple transactions.
rho_class, p_class = spearmanr(
    noise_data["random_noise"], noise_data[TARGET]
)
rho_spend, p_spend = spearmanr(
    noise_returned["random_noise"], noise_returned[SPEND]
)

_, p_u = mannwhitneyu(
    noise_data.loc[noise_data[TARGET].eq(1), "random_noise"],
    noise_data.loc[noise_data[TARGET].eq(0), "random_noise"],
    alternative="two-sided"
)

results = pd.DataFrame({
    "Target": ["Repeat purchase", "Spending among returners"],
    "Rows checked": [len(noise_data), len(noise_returned)],
    "Spearman correlation": [rho_class, rho_spend],
    "p-value": [p_class, p_spend]
}).set_index("Target")

display(results.round(3))

print(f"Mann–Whitney p-value between classes: {p_u:.3g}")
print(
    f"Noise mean: {noise_data['random_noise'].mean():.3f} | "
    f"standard deviation: {noise_data['random_noise'].std():.3f}"
)

# Keep random_noise until models are compared with and without it
# using the same validation splits for classification and regression.

,Rows checked,Spearman correlation,p-value
Target,,,
Repeat purchase,6469,-0.008,0.510
Spending among returners,3560,0.014,0.396


Mann–Whitney p-value between classes: 0.51
Noise mean: -0.022 | standard deviation: 1.001


`random_noise` showed **no clear association with either target**:

- **Repeat purchases:** the class distributions almost overlapped, and repeat rates stayed close to the overall 56% without a clear pattern. Spearman correlation was **−0.008** (p = **0.510**).
- **Spending among returners:** spending distributions were similar across bins. Spearman correlation was **0.014** (p = **0.396**).

These results suggest little standalone predictive value, making `random_noise` a **candidate for removal**.

We kept the column for feature selection. We will compare models with and without it using the same validation splits for both tasks before deciding whether to remove it. These exploratory checks cannot rule out nonlinear effects or interactions.

## 7. Shorter category labels

The categorical values are anonymised: each column has one fixed prefix plus a number (e.g. `Maybe Tomorrow Payment 7`). The prefix carries no information, so we keep only the number.

The number stays **text** (a category), not an integer: payment type 9 is not "bigger" than type 7.

In [95]:
cat_cols = column_groups["categoricals"]

print("Before:", df[cat_cols].head(1).to_dict("records"))

for data in (df, test):
    for col in cat_cols:
        values = data[col].str.strip()
        shortened = values.str.extract(r"(\d+)$", expand=False)

        invalid = values.notna() & shortened.isna()
        if invalid.any():
            raise ValueError(f"Unexpected category format in {col}")

        data[col] = shortened

print("After:", df[cat_cols].head(1).to_dict("records"))

Before: [{'payment_type': 'Maybe Tomorrow Payment 10', 'salesperson': 'VLOOKUP Master 9', 'commercial_zone': 'Decorative Dashboard Zone 4', 'transport_method': 'Maybe Tomorrow Delivery 61', 'warehouse': 'Final Excel Warehouse 1', 'document_series': 'Ctrl Z Series 2'}]
After: [{'payment_type': '10', 'salesperson': '9', 'commercial_zone': '4', 'transport_method': '61', 'warehouse': '1', 'document_series': '2'}]


### Key insight - Section 7

**Found:** every categorical column has a single prefix, with no spelling variants (extra spaces, different capitals), so there is nothing to merge.

**For preprocessing:** keep only the number, as text. Encode these columns as categories (one-hot or target encoding), never as numbers.

In [97]:
from pathlib import Path

# Save the completed cleaning steps for notebooks 02 and 03.
# Keep random_noise for later model comparison.
train_clean = df.copy()
test_clean = test.copy()

processed_dir = Path(DATA_DIR) / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

train_clean.to_csv(processed_dir / "train_clean.csv", index=False)
test_clean.to_csv(processed_dir / "test_clean.csv", index=False)

missing_spend = (
    train_clean[TARGET].eq(1)
    & train_clean[SPEND].isna()
)

print("Saved:", train_clean.shape, test_clean.shape)
print("Returning purchases still without spending:", missing_spend.sum())

Saved: (6469, 40) (986, 38)
Returning purchases still without spending: 61


### Summary of decisions

| # | Issue | Rows | Decision | Where |
|---|---|---|---|---|
| 2 | Empty rows (only ID, customer, target) | 65 | drop | train only |
| 3.1 | `prior_mean_purchase_value`: `1e9`, sign flip, too large, missing | 272 | recompute total / count | train + test |
| 3.2 | `prior_max_purchase_value` > total spend | 45 | rebuild from earlier purchases (train); NaN, impute (test) | train + test |
| 3.3 | `spend_365d`: `1e9`, sign flip, > total spend | 69 | remove sign; rebuild from dates (train), else NaN, impute | train + test |
| 3.4 | -4 in `days_since_previous_purchase` | 16 | rebuild from dates (train); NaN, impute (test) | train + test |
| 3.4 | -4 in `prior_mean_gap_days` | 16 | rebuild from dates (train); NaN, impute (test) | train + test |
| 5 | Returners without spend | 196 | recover 176 from train, drop 20 | regression target |
| 6 | `random_noise` | all | drop | train + test |
| 7 | Anonymised category labels (`Maybe Tomorrow Payment 7`) | all | keep only the number, as text | train + test |

**Next:** `02_eda.ipynb` explores the clean data saved above, and `03_feature_engineering_selection.ipynb` builds the features.